In [2]:
import pandas as pd
import seaborn as sns
import numpy as np
from scipy.stats.contingency import crosstab, chi2_contingency
from scipy.stats import ranksums, iqr
import pingouin as pg
import warnings
import scorecardpy as sc
import workflow_utils.visualization as viz
import workflow_utils.stats as stats

In [3]:
warnings.simplefilter(action='ignore', category=FutureWarning)
pd.options.display.float_format = '{:.3f}'.format

In [41]:
accepted = pd.read_csv('data/accepted.csv')
accepted.head()

C:\Users\ivan\AppData\Local\Temp\ipykernel_18940\2360842760.py:1: DtypeWarning: Columns (0,19,49,59,118,129,130,131,134,135,136,139,145,146,147) have mixed types. Specify dtype option on import or set low_memory=False.
  accepted = pd.read_csv('data/accepted.csv')


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.000,3600.000,3600.000,36 months,13.990,123.030,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.000,24700.000,24700.000,36 months,11.990,820.280,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.000,20000.000,20000.000,60 months,10.780,432.660,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.000,35000.000,35000.000,60 months,14.850,829.900,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.000,10400.000,10400.000,60 months,22.450,289.910,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [42]:
accepted.shape

(2260701, 151)

Drop columns with > 30% missing or 99% values in a single value

In [43]:
percent_missing = accepted.isna().mean()


In [44]:
accepted = accepted.drop(columns=percent_missing[percent_missing > 0.3].index)

In [45]:
accepted.shape

(2260701, 93)

In [46]:
mode_proportions = accepted.apply(lambda x: x.value_counts(normalize=True).iloc[0])
mode_proportions

id                           0.000
loan_amnt                    0.083
funded_amnt                  0.083
funded_amnt_inv              0.079
term                         0.712
                              ... 
total_bc_limit               0.011
total_il_high_credit_limit   0.120
hardship_flag                1.000
disbursement_method          0.965
debt_settlement_flag         0.985
Length: 93, dtype: float64

In [47]:
accepted = accepted.drop(columns=mode_proportions[mode_proportions >= 0.95].index)
accepted.shape

(2260701, 80)

In [48]:
accepted.columns

Index(['id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate',
       'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length',
       'home_ownership', 'annual_inc', 'verification_status', 'issue_d',
       'loan_status', 'url', 'purpose', 'title', 'zip_code', 'addr_state',
       'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low',
       'fico_range_high', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal',
       'revol_util', 'total_acc', 'initial_list_status', 'out_prncp',
       'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp',
       'total_rec_int', 'recoveries', 'collection_recovery_fee',
       'last_pymnt_d', 'last_pymnt_amnt', 'last_credit_pull_d',
       'last_fico_range_high', 'last_fico_range_low', 'application_type',
       'tot_coll_amt', 'tot_cur_bal', 'total_rev_hi_lim',
       'acc_open_past_24mths', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util',
       'mo_sin_old_il_acct', 'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_t

In [49]:
columns_todrop = ['id', 'sub_grade', 'url', 'title', 'emp_title', 'zip_code']
accepted = accepted.drop(columns=columns_todrop)

In [50]:
accepted.dtypes.unique()

array([dtype('float64'), dtype('O')], dtype=object)

In [51]:
datetime_columns = ["issue_d", "earliest_cr_line", "last_pymnt_d", "last_credit_pull_d"]
accepted[datetime_columns].head()

,issue_d,earliest_cr_line,last_pymnt_d,last_credit_pull_d
0,Dec-2015,Aug-2003,Jan-2019,Mar-2019
1,Dec-2015,Dec-1999,Jun-2016,Mar-2019
2,Dec-2015,Aug-2000,Jun-2017,Mar-2019
3,Dec-2015,Sep-2008,Feb-2019,Mar-2019
4,Dec-2015,Jun-1998,Jul-2016,Mar-2018


In [53]:
accepted[datetime_columns] = accepted[datetime_columns].apply(pd.to_datetime, format='%b-%Y')

In [54]:
accepted[datetime_columns].head()

,issue_d,earliest_cr_line,last_pymnt_d,last_credit_pull_d
0,2015-12-01,2003-08-01,2019-01-01,2019-03-01
1,2015-12-01,1999-12-01,2016-06-01,2019-03-01
2,2015-12-01,2000-08-01,2017-06-01,2019-03-01
3,2015-12-01,2008-09-01,2019-02-01,2019-03-01
4,2015-12-01,1998-06-01,2016-07-01,2018-03-01


The column of interest should be either num_accts_ever_120_pd or loan_status or delinq_2yrs or num_tl_90g_dpd_24m

In [55]:
accepted["loan_status"].unique()

array(['Fully Paid', 'Current', 'Charged Off', 'In Grace Period',
       'Late (31-120 days)', 'Late (16-30 days)', 'Default', nan,
       'Does not meet the credit policy. Status:Fully Paid',
       'Does not meet the credit policy. Status:Charged Off'],
      dtype=object)

In [61]:
accepted["num_accts_ever_120_pd"].unique()

array([ 2.,  0.,  1.,  4., 12.,  5.,  3.,  7.,  8.,  6., 17., 13., 11.,
       18.,  9., 26., 10., 14., 16., 25., 32., 23., 15., 20., 24., 30.,
       27., 21., 22., 19., 31., 29., 39., 28., 35., nan, 34., 36., 37.,
       33., 45., 58., 42., 38., 51.])

In [62]:
accepted["delinq_2yrs"].unique()

array([ 0.,  1.,  2.,  3.,  4.,  6.,  5., 15.,  7.,  9., 10.,  8., 11.,
       13., 14., 12., 30., 18., 16., 17., 26., 20., 19., 22., 27., 39.,
       nan, 28., 21., 25., 32., 58., 24., 35., 42., 23., 29., 36.])

In [63]:
accepted["num_tl_90g_dpd_24m"].unique()

array([ 0.,  1.,  2.,  4., 12.,  3.,  7.,  8.,  5., 13., 11., 10.,  6.,
        9., 14., 30., 16., 15., 26., 22., 18., 39., 20., 19., 17., nan,
       21., 58., 35., 42., 25., 23., 24., 36., 29.])

Drop columns that would not be available for an application scorecard

In [ ]:
columns_todrop = ["grade", ]